In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pybaseball import statcast

In [ ]:
spark = (
    SparkSession.builder
    .appName("dataset_creation")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .getOrCreate()
)

In [ ]:
# completed regular seasons, excluding the current season
seasons = {
    2018: ("2018-03-29", "2018-10-01"),
    2019: ("2019-03-20", "2019-09-29"),
    2020: ("2020-07-23", "2020-09-27"),
    2021: ("2021-04-01", "2021-10-03"),
    2022: ("2022-04-07", "2022-10-05"),
    2023: ("2023-03-30", "2023-10-01"),
    2024: ("2024-03-20", "2024-09-30"),
    2025: ("2025-03-18", "2025-09-28"),
}

Path("./data/raw").mkdir(parents=True, exist_ok=True)

# save one season at a time so pandas does not hold the entire dataset
for year, (start, end) in seasons.items():
    df = statcast(start_dt=start, end_dt=end)
    df = df[df["game_type"] == "R"]
    df.to_parquet(f"./data/raw/statcast_{year}.parquet", index=False)
    print(f"{year}: {len(df):,} observations")

In [ ]:
paths = [f"./data/raw/statcast_{year}.parquet" for year in seasons]
throws_df = spark.read.option("mergeSchema", "true").parquet(*paths)
throws_df = throws_df.filter(F.col("game_type") == "R")

throws_df.groupBy("game_year").count().orderBy("game_year").show()
print(f"Total observations: {throws_df.count():,}")

In [ ]:
throws_df.write.mode("overwrite").parquet("./data/full_dataset.parquet")